<a href="https://colab.research.google.com/github/ideolixlearninghub/Assessment-/blob/main/my_exam_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
pip install flask flask-cors flask-jwt-extended

In [7]:
import os

# Create folders
os.makedirs("backend/routes", exist_ok=True)
os.makedirs("backend/utils", exist_ok=True)

# Create empty init files
open("backend/routes/__init__.py", "w").close()
open("backend/utils/__init__.py", "w").close()

# Create main backend files
files = {
    "backend/app.py": "",
    "backend/config.py": "",
    "backend/database.py": "",
    "backend/models.py": "",
    "backend/routes/auth.py": "",
    "backend/routes/subjects.py": "",
    "backend/routes/questions.py": "",
    "backend/routes/assessment.py": "",
    "backend/routes/certificate.py": "",
    "backend/utils/jwt_helper.py": "",
    "backend/utils/question_generator.py": "",
    "backend/utils/certificate_generator.py": "",
}

for path, content in files.items():
    with open(path, "w") as f:
        f.write(content)

"Backend file structure created successfully."

'Backend file structure created successfully.'

In [19]:
app_code = """
from flask import Flask
from flask_cors import CORS
from flask_jwt_extended import JWTManager

from routes.auth import auth_bp
from routes.subjects import subjects_bp
from routes.questions import questions_bp
from routes.assessment import assessment_bp
from routes.certificate import certificate_bp

app = Flask(__name__)

# Config
app.config['JWT_SECRET_KEY'] = 'ideolix_secret_key'
app.config['JSON_SORT_KEYS'] = False

# CORS + JWT
CORS(app)
jwt = JWTManager(app)

# Register routes
app.register_blueprint(auth_bp, url_prefix='/auth')
app.register_blueprint(subjects_bp, url_prefix='/subjects')
app.register_blueprint(questions_bp, url_prefix='/questions')
app.register_blueprint(assessment_bp, url_prefix='/assessment')
app.register_blueprint(certificate_bp, url_prefix='/certificate')


@app.route('/')
def home():
    return {"message": "Ideolix Backend Running Successfully"}


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""

with open("backend/app.py", "w") as f:
    f.write(app_code)

"app.py written successfully."


'app.py written successfully.'

In [8]:
app_code = """
from flask import Flask
from flask_cors import CORS
from flask_jwt_extended import JWTManager

from routes.auth import auth_bp
from routes.subjects import subjects_bp
from routes.questions import questions_bp
from routes.assessment import assessment_bp
from routes.certificate import certificate_bp

from config import Config
from database import init_db

def create_app():
    app = Flask(__name__)
    app.config.from_object(Config)

    CORS(app)
    JWTManager(app)
    init_db()

    # Register Blueprints
    app.register_blueprint(auth_bp, url_prefix="/auth")
    app.register_blueprint(subjects_bp, url_prefix="/subjects")
    app.register_blueprint(questions_bp, url_prefix="/questions")
    app.register_blueprint(assessment_bp, url_prefix="/assessment")
    app.register_blueprint(certificate_bp, url_prefix="/certificate")

    @app.route("/")
    def home():
        return {"status": "Ideolix Backend Running"}

    return app

app = create_app()

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
"""

with open("backend/app.py", "w") as f:
    f.write(app_code)

"backend/app.py created."

'backend/app.py created.'

In [21]:
config_code = """
import os

class Config:
    SECRET_KEY = 'supersecretkey_ideolix'
    JWT_SECRET_KEY = 'jwtsecret_ideolix'
    SQLALCHEMY_DATABASE_URI = 'sqlite:///ideolix.db'
    SQLALCHEMY_TRACK_MODIFICATIONS = False
"""

with open("backend/config.py", "w") as f:
    f.write(config_code)

"backend/config.py created."


'backend/config.py created.'

In [9]:
config_code = """
import os

class Config:
    SECRET_KEY = 'ideolixsecretkey'
    JWT_SECRET_KEY = 'ideolixjwtsecret'
    SQLALCHEMY_DATABASE_URI = 'sqlite:///ideolix.db'
    SQLALCHEMY_TRACK_MODIFICATIONS = False
"""

with open("backend/config.py", "w") as f:
    f.write(config_code)

"backend/config.py created."

'backend/config.py created.'

In [10]:
database_code = """
from flask_sqlalchemy import SQLAlchemy
from app import app

db = SQLAlchemy()

def init_db():
    db.init_app(app)
    with app.app_context():
        db.create_all()
"""

with open("backend/database.py", "w") as f:
    f.write(database_code)

"backend/database.py created."

'backend/database.py created.'

In [11]:
models_code = """
from database import db
from datetime import datetime
import uuid

class User(db.Model):
    id = db.Column(db.String, primary_key=True, default=lambda: str(uuid.uuid4()))
    full_name = db.Column(db.String(100), nullable=False)
    email = db.Column(db.String(100), unique=True, nullable=False)
    password_hash = db.Column(db.String(200), nullable=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

class Question(db.Model):
    id = db.Column(db.String, primary_key=True, default=lambda: str(uuid.uuid4()))
    subject = db.Column(db.String(50), nullable=False)
    curriculum = db.Column(db.String(50), nullable=False)
    grade = db.Column(db.String(50), nullable=False)
    topic = db.Column(db.String(50), nullable=True)
    type = db.Column(db.String(20), nullable=False)  # MCQ, Numeric, Essay
    body = db.Column(db.String, nullable=False)
    options = db.Column(db.String, nullable=True)  # JSON string
    correct_answer = db.Column(db.String, nullable=True)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

class Assessment(db.Model):
    id = db.Column(db.String, primary_key=True, default=lambda: str(uuid.uuid4()))
    user_id = db.Column(db.String, db.ForeignKey('user.id'), nullable=False)
    score = db.Column(db.Float, nullable=True)
    passed = db.Column(db.Boolean, default=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)

class Certificate(db.Model):
    id = db.Column(db.String, primary_key=True, default=lambda: str(uuid.uuid4()))
    user_id = db.Column(db.String, db.ForeignKey('user.id'), nullable=False)
    assessment_id = db.Column(db.String, db.ForeignKey('assessment.id'), nullable=False)
    score = db.Column(db.Float, nullable=False)
    created_at = db.Column(db.DateTime, default=datetime.utcnow)
"""

with open("backend/models.py", "w") as f:
    f.write(models_code)

"Models created successfully."


'Models created successfully.'

In [12]:
# --------------------------
# backend/utils/question_generator.py
# --------------------------
question_generator_code = """
import random
import uuid

subjects = [
    'Mathematics', 'English', 'Physics', 'Chemistry', 'Biology',
    'History', 'Geography', 'Computer Science', 'Economics',
    'Civic Education', 'Government', 'Business Studies', 'Accounting',
    'Religious Studies', 'Health Education', 'Home Economics', 'Agricultural Science'
]

def generate_question(curriculum, grade, subject=None):
    if not subject:
        subject = random.choice(subjects)
    topics = [f'Topic {i}' for i in range(1, 6)]
    topic = random.choice(topics)
    options = ['A', 'B', 'C', 'D']
    correct = random.choice(options)
    return {
        "id": str(uuid.uuid4()),
        "curriculum": curriculum,
        "grade": grade,
        "subject": subject,
        "topic": topic,
        "question": f"Solve this {subject} question on {topic} (Grade {grade})",
        "options": options,
        "correct_answer": correct
    }
"""

with open("backend/utils/question_generator.py", "w") as f:
    f.write(question_generator_code)

# --------------------------
# backend/utils/certificate_generator.py
# --------------------------
certificate_generator_code = """
import uuid
from datetime import datetime

def generate_certificate(user_id, assessment_id, score):
    return {
        "certificate_id": str(uuid.uuid4()),
        "user_id": user_id,
        "assessment_id": assessment_id,
        "score": score,
        "date": datetime.utcnow().strftime('%Y-%m-%d')
    }
"""

with open("backend/utils/certificate_generator.py", "w") as f:
    f.write(certificate_generator_code)

"Utility files created."


'Utility files created.'

In [13]:
# backend/routes/auth.py
auth_code = """
from flask import Blueprint, request, jsonify
from werkzeug.security import generate_password_hash, check_password_hash
from flask_jwt_extended import create_access_token, jwt_required, get_jwt_identity
from models import User
from database import db

auth_bp = Blueprint('auth', __name__)

@auth_bp.route('/register', methods=['POST'])
def register():
    data = request.json
    full_name = data.get('full_name')
    email = data.get('email')
    password = data.get('password')

    if User.query.filter_by(email=email).first():
        return jsonify({"msg": "Email already registered"}), 400

    user = User(full_name=full_name, email=email, password_hash=generate_password_hash(password))
    db.session.add(user)
    db.session.commit()
    return jsonify({"msg": "User registered successfully"}), 201

@auth_bp.route('/login', methods=['POST'])
def login():
    data = request.json
    email = data.get('email')
    password = data.get('password')

    user = User.query.filter_by(email=email).first()
    if not user or not check_password_hash(user.password_hash, password):
        return jsonify({"msg": "Invalid credentials"}), 401

    access_token = create_access_token(identity=user.id)
    return jsonify({"access_token": access_token})

@auth_bp.route('/profile', methods=['GET'])
@jwt_required()
def profile():
    user_id = get_jwt_identity()
    user = User.query.get(user_id)
    return jsonify({
        "id": user.id,
        "full_name": user.full_name,
        "email": user.email
    })
"""

with open("backend/routes/auth.py", "w") as f:
    f.write(auth_code)

"Authentication routes created."


'Authentication routes created.'

In [14]:
# backend/routes/subjects.py
subjects_code = """
from flask import Blueprint, jsonify

subjects_bp = Blueprint('subjects', __name__)

curricula = {
    "nigerian": {
        "grade1": ["Mathematics", "English", "Basic Science", "Social Studies", "Civic Education"],
        "grade2": ["Mathematics", "English", "Basic Science", "Computer Studies", "Agricultural Science"],
        "grade12": ["Mathematics", "English", "Physics", "Chemistry", "Biology", "Further Maths"]
    },
    "british": {
        "year1": ["Maths", "English", "Science"],
        "year13": ["Maths", "English", "Biology", "Physics", "Chemistry"]
    },
    "american": {
        "grade1": ["Math", "English", "Science", "Social Studies"],
        "grade12": ["Math", "English", "Biology", "Physics", "Chemistry"]
    }
}

@subjects_bp.route("/curricula", methods=["GET"])
def get_curricula():
    return jsonify(list(curricula.keys()))

@subjects_bp.route("/grades/<curriculum>", methods=["GET"])
def get_grades(curriculum):
    return jsonify(list(curricula.get(curriculum, {}).keys()))

@subjects_bp.route("/subjects/<curriculum>/<grade>", methods=["GET"])
def get_subjects(curriculum, grade):
    return jsonify(curricula.get(curriculum, {}).get(grade, []))
"""

with open("backend/routes/subjects.py", "w") as f:
    f.write(subjects_code)

"Subjects routes created."


'Subjects routes created.'

In [15]:
# backend/routes/questions.py
questions_code = """
from flask import Blueprint, request, jsonify
from utils.question_generator import generate_question

questions_bp = Blueprint('questions', __name__)

@questions_bp.route("/practice", methods=["POST"])
def practice():
    data = request.json
    curriculum = data.get("curriculum")
    grade = data.get("grade")
    subject = data.get("subject")

    questions = [generate_question(curriculum, grade, subject) for _ in range(10)]
    return jsonify(questions)
"""

with open("backend/routes/questions.py", "w") as f:
    f.write(questions_code)

"Practice questions routes created."


'Practice questions routes created.'

In [16]:
# backend/routes/assessment.py
assessment_code = """
from flask import Blueprint, request, jsonify
from utils.question_generator import generate_question
import uuid

assessment_bp = Blueprint('assessment', __name__)

@assessment_bp.route("/generate", methods=["POST"])
def generate_assessment():
    data = request.json
    curriculum = data.get("curriculum")
    grade = data.get("grade")
    subject = data.get("subject")

    questions = [generate_question(curriculum, grade, subject) for _ in range(70)]
    return jsonify({
        "assessment_id": str(uuid.uuid4()),
        "questions": questions
    })

@assessment_bp.route("/submit", methods=["POST"])
def submit_assessment():
    data = request.json
    total_correct = data.get("total_correct", 0)

    score_percent = (total_correct / 70) * 100
    passed = score_percent >= 85
    certificate_id = str(uuid.uuid4()) if passed else None

    return jsonify({
        "score": score_percent,
        "passed": passed,
        "certificate_id": certificate_id
    })
"""

with open("backend/routes/assessment.py", "w") as f:
    f.write(assessment_code)

"Assessment routes created."


'Assessment routes created.'

In [17]:
# backend/routes/certificate.py
certificate_code = """
from flask import Blueprint, request, jsonify
from utils.certificate_generator import generate_certificate

certificate_bp = Blueprint('certificate', __name__)

@certificate_bp.route("/generate", methods=["POST"])
def generate_cert():
    data = request.json
    user_id = data.get("user_id")
    assessment_id = data.get("assessment_id")
    score = data.get("score")

    certificate = generate_certificate(user_id, assessment_id, score)
    return jsonify(certificate)
"""

with open("backend/routes/certificate.py", "w") as f:
    f.write(certificate_code)

"Certificate routes created."


'Certificate routes created.'

In [19]:
!pip install flask flask-cors flask-jwt-extended

from pyngrok import ngrok
import subprocess
import sys

# Set your ngrok authtoken here. Get it from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("35ZJKgPZxlIZomIYJ3re1JmoqKO_C3U81bKyuy5hgriA7tiv")

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print("Public backend URL:", public_url)

# Diagnostic: Check current directory and backend folder contents
!pwd
!ls -l backend/

# Start the Flask app as a background process
# Use sys.executable to ensure the current Python interpreter is used
flask_process = subprocess.Popen([sys.executable, 'backend/app.py'])
print(f"Flask app started with PID: {flask_process.pid}")

Public backend URL: NgrokTunnel: "https://morris-untapered-judson.ngrok-free.dev" -> "http://localhost:5000"
/content
total 24
-rw-r--r-- 1 root root 1029 Nov 16 16:55 app.py
-rw-r--r-- 1 root root  198 Nov 16 16:55 config.py
-rw-r--r-- 1 root root  169 Nov 16 16:55 database.py
-rw-r--r-- 1 root root 1806 Nov 16 16:55 models.py
drwxr-xr-x 2 root root 4096 Nov 16 16:55 routes
drwxr-xr-x 2 root root 4096 Nov 16 16:55 utils
Flask app started with PID: 5066
